In [ ]:
import json
import numpy as np
import pandas as pd
import anndata
import logging
import requests

logging.basicConfig(level=logging.INFO)

In [ ]:
logging.info("Loading embedding data")
adata = anndata.read_h5ad(snakemake.input.embedding_adata)
logging.info("Loading read count data")
read_count_adata = anndata.read_h5ad(snakemake.input.read_count_adata, backed="r")
assert (adata.obs.orig_ids == read_count_adata.obs.index).all()
adata.obs["index_int"] = list(range(len(adata.obs)))

In [ ]:
API_URL = snakemake.params.api_url
MODEL_NAME = snakemake.params.model_name

# Construct the prompt in the same format as the local LLaVA notebook
# The prompt follows the Mistral instruct format used by the LLaVA model
PROMPT = (
    "[INST] Help me analyzing this sample of cells. "
    "Respond in proper English sentences in a tone of uncertainty and focus on "
    "the biology of the sample rather than any potential donor or patient information "
    "(e.g. do not mention age and sex) [/INST] "
    "Sure. I'll respond as you requested, focusing on the sample of cells and "
    "avoiding patient or donor information. </s>"
    "[INST] Describe the biological state of these cells\n<image> [/INST]"
)


def annotate_cluster_via_api(embedding):
    """Send a cluster's average embedding to the CellWhisperer LLaVA API and return the generated text."""
    pload = {
        "model": MODEL_NAME,
        "prompt": PROMPT,
        "temperature": 0.0,
        "top_p": 1.0,
        "max_new_tokens": 256,
        "stop": "</s>",
        "images": [embedding.tolist()],
    }
    response = requests.post(
        API_URL + "/worker_generate_stream",
        headers={"User-Agent": "LLaVA Client"},
        json=pload,
        stream=True,
        timeout=120,
    )
    response.raise_for_status()
    output = ""
    for chunk in response.iter_lines(decode_unicode=False, delimiter=b"\0"):
        if chunk:
            data = json.loads(chunk.decode())
            if data["error_code"] == 0:
                output = data["text"][len(PROMPT):].strip()
            else:
                raise RuntimeError(f"API error: {data}")
    return output


def cluster_annotation(cluster_values):
    grouped_embeddings = adata.obs.groupby(cluster_values, observed=True).apply(
        lambda group: adata.X[group.index_int].mean(axis=0), include_groups=False
    )
    cluster_labels = {}
    for cluster_id, grouped_embedding in grouped_embeddings.items():
        logging.info(f"Annotating cluster {cluster_id} via API")
        generated_text = annotate_cluster_via_api(np.asarray(grouped_embedding).ravel())
        print(f"{cluster_id}: {generated_text[:80]}...")
        cluster_labels[cluster_id] = generated_text
    return pd.Series(cluster_labels)

In [ ]:
dfs = []
cluster_series = {"leiden": adata.obs["leiden"].values}
try:
    for cluster_field in read_count_adata.uns["cluster_fields"]:
        cluster_series[cluster_field] = read_count_adata.obs[cluster_field].values
except KeyError:
    pass

logging.info("Starting cluster annotation via API")
for cluster_field, cluster_values in cluster_series.items():
    logging.info(f"Annotating clusters for field: {cluster_field}")
    out = cluster_annotation(cluster_values)
    out.index.name = "cluster_values"
    out.name = "cluster_annotations"
    out = out.to_frame()
    out["cluster_field"] = cluster_field
    dfs.append(out)

In [ ]:
pd.concat(dfs).to_csv(snakemake.output.csv)